# Carga Gold

Lê as duas tabelas silver e constrói o esquema estrela da camada gold. As medidas (sets,
duração, seeds, estatísticas de jogo, idade e altura na partida) ficam em dois fatos, um
por grão de evento: `fato_partida`, uma linha por partida, e `fato_atleta_partida`, uma linha
por atleta em cada lado de uma partida. O contexto que descreve esses eventos fica em quatro
dimensões: `dim_data` (o calendário), `dim_torneio` (a edição), `dim_atleta` (a pessoa,
deduplicada a partir das participações) e `dim_fase` (a chave do torneio e a fase que ela
representa). Cada tabela existe para responder pelo menos uma das seis perguntas do objetivo.
As chaves são substitutas e determinísticas, e a integridade entre fatos e dimensões é
verificada na última seção.

In [0]:
from pyspark.sql import functions as F

### Leitura da Silver:
p  = spark.table("workspace.silver.partida")
ap = spark.table("workspace.silver.atleta_partida")

## 1. Dimensões

### 1.1. dim_data

Um dia por linha, da primeira à última partida. `id_data` é a própria data como inteiro
`yyyyMMdd` — legível e determinística, convenção de dimensão de calendário.

In [0]:
# Ciclo olímpico: os quatro anos que terminam em cada Olimpíada
ciclo_olimpico = (
    F.when(F.col("ano") <= 2000, "Sydney-2000")     # o dado começa nos Jogos, único torneio de 2000
     .when(F.col("ano") <= 2004, "Atenas-2004")
     .when(F.col("ano") <= 2008, "Pequim-2008")
     .when(F.col("ano") <= 2012, "Londres-2012")
     .when(F.col("ano") <= 2016, "Rio-2016")
     .otherwise("Tóquio-2020")                       # incompleto: o dado acaba em 2019
)

limites = p.agg(F.min("data").alias("inicio"), F.max("data").alias("fim")).first()

# Um dia por linha, do primeiro ao último dia com partida
dim_data = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{limites.inicio}'), to_date('{limites.fim}'), interval 1 day)) AS data
    """)
    .select(
        F.date_format("data", "yyyyMMdd").cast("int").alias("id_data"),
        "data",
        F.year("data").alias("ano"),
        F.month("data").alias("mes"),
    )
    .withColumn("ciclo_olimpico", ciclo_olimpico)
)

(dim_data.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_data"))

display(spark.table("workspace.gold.dim_data").limit(5))

### 1.2. dim_torneio

Um torneio por linha: circuito, cidade-sede, país, ano e gênero, com as datas da primeira
e da última partida. A fonte identifica o torneio pela cidade e pelo ano; em três casos
(Rio de Janeiro 2016, Ljubljana 2018 e Sydney 2017 feminino) a mesma cidade sediou dois
eventos no mesmo ano, que aqui ficam fundidos — nenhuma das perguntas depende dessa separação.

In [0]:
chave_torneio = ["circuito", "torneio", "pais_torneio", "ano", "genero"]

dim_torneio = (
    p.groupBy(*chave_torneio)
     .agg(F.min("data").alias("data_inicio"), F.max("data").alias("data_fim"))
     # id_torneio: hash da chave natural, mesmo critério dos ids da silver
     .withColumn("id_torneio", F.xxhash64(*chave_torneio).bitwiseAND(F.lit(9223372036854775807)))
     .select(
         "id_torneio",
         "circuito",
         F.col("torneio").alias("cidade"),
         F.col("pais_torneio").alias("pais"),
         "ano", "genero",
         "data_inicio", "data_fim",
     )
)

(dim_torneio.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_torneio"))

display(spark.table("workspace.gold.dim_torneio").limit(5))

### 1.3. dim_atleta

Um atleta por linha, deduplicado das participações por `id_atleta`. Os atributos que podem
variar entre partidas (nome, gênero, altura, país) ficam com o valor mais frequente; a
carreira é o intervalo entre a primeira e a última partida registradas.

In [0]:
dim_atleta = (
    ap.groupBy("id_atleta")
      .agg(
          F.mode("nome").alias("nome"),
          F.min("nascimento").alias("nascimento"),   # constante por id, faz parte do hash
          F.mode("genero").alias("genero"),
          F.mode("altura_cm").alias("altura_cm"),    # mode ignora NULL
          F.mode("pais").alias("pais"),              # país mais frequente; 1 atleta trocou de federação
          F.min("data").alias("primeira_partida"),
          F.max("data").alias("ultima_partida"),
      )
      .withColumn("flag_sem_nascimento", F.col("nascimento").isNull())
)

(dim_atleta.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_atleta"))

display(spark.table("workspace.gold.dim_atleta").limit(5))

### 1.4. dim_fase

Uma chave de torneio por linha, com a fase que ela representa. O agrupamento dos 36
valores de `bracket` em três fases é decisão de modelagem; aqui ela vira uma tabela que se
lê, e não uma regra escondida no tratamento.

In [0]:
dim_fase = (
    p.select("chave", "fase").distinct()
     .withColumn("id_fase", F.xxhash64("chave").bitwiseAND(F.lit(9223372036854775807)))
     .select("id_fase", "chave", "fase")
)

(dim_fase.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_fase"))

display(spark.table("workspace.gold.dim_fase").limit(5))

## 2. Fatos

### 2.1. fato_partida

Uma partida por linha, com as chaves para torneio, data e fase, o resultado decomposto, os
seeds das duas duplas e as flags. `rodada` e `num_partida` ficam no próprio fato: identificam
a partida dentro da chave, mas não têm atributos que justifiquem uma dimensão.

In [0]:
fato_partida = p.select(
    # Chaves; id_torneio e id_fase com o mesmo hash das dimensões
    "id_partida",
    F.xxhash64(*chave_torneio).bitwiseAND(F.lit(9223372036854775807)).alias("id_torneio"),
    F.date_format("data", "yyyyMMdd").cast("int").alias("id_data"),
    F.xxhash64("chave").bitwiseAND(F.lit(9223372036854775807)).alias("id_fase"),
    # Identificação da partida dentro da chave
    "rodada", "num_partida",
    # Resultado
    "sets_vencedor", "sets_perdedor",
    (F.col("sets_vencedor") + F.col("sets_perdedor")).alias("total_sets"),
    # Pontos somados do placar; nulo quando o placar é irregular
    F.when(F.col("placar_regular"),
           F.expr("aggregate(transform(split(placar, ', '), s -> int(split(s, '-')[0]) + int(split(s, '-')[1])), 0, (a, x) -> a + x)")
    ).alias("total_pontos"),
    "duracao_min",
    # Seeds
    "w_seed_principal", "w_seed_qualificatoria",
    "l_seed_principal", "l_seed_qualificatoria",
    # Flags
    "flag_partida_incompleta", "flag_duracao_suspeita", "flag_placar_inconsistente",
    # Flag Zebra: o pior ranqueado venceu. Seed principal quando as duas duplas têm; da qualificatória quando só têm esse; nulo sem seed
    F.when(F.col("w_seed_principal").isNotNull() & F.col("l_seed_principal").isNotNull(),
            F.col("w_seed_principal") > F.col("l_seed_principal"))
    .when(F.col("w_seed_qualificatoria").isNotNull() & F.col("l_seed_qualificatoria").isNotNull(),
            F.col("w_seed_qualificatoria") > F.col("l_seed_qualificatoria"))
    .alias("flag_zebra"),
)

(fato_partida.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.fato_partida"))

display(spark.table("workspace.gold.fato_partida").limit(5))

### 2.2. fato_atleta_partida

Um atleta em um lado de uma partida, com as chaves para partida, atleta, torneio e data, a
idade naquela partida, as estatísticas de jogo e as flags. `flag_em_casa` compara o país do
atleta naquela participação com o país-sede do torneio.

In [0]:
estatisticas = ["tot_attacks", "tot_kills", "tot_errors", "tot_hitpct",
                "tot_aces", "tot_serve_errors", "tot_blocks", "tot_digs"]

# O país-sede e a chave do torneio vêm da partida; genero sai da participação para não duplicar no join
fato_atleta_partida = (
    ap.drop("genero")
      .join(p.select("id_partida", *chave_torneio), "id_partida")
      .select(
          # Chaves
          "id_partida", "id_atleta", "vencedor",
          F.xxhash64(*chave_torneio).bitwiseAND(F.lit(9223372036854775807)).alias("id_torneio"),
          F.date_format("data", "yyyyMMdd").cast("int").alias("id_data"),
          # Atleta naquela partida
          "idade_na_partida",
          F.coalesce(F.col("pais") == F.col("pais_torneio"), F.lit(False)).alias("flag_em_casa"),
          # Estatísticas de jogo
          "tem_estatistica", *estatisticas,
          # Flags
          "flag_estatistica_invalida", "flag_idade_atipica",
      )
)

(fato_atleta_partida.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.fato_atleta_partida"))

display(spark.table("workspace.gold.fato_atleta_partida").limit(5))

## 3. Validação pós-carga

Confere as chaves primárias de cada tabela, que nenhuma chave estrangeira aponta para linha
inexistente e que os fatos têm o mesmo número de linhas que as tabelas silver de origem.

In [0]:
g = {t: spark.table(f"workspace.gold.{t}") for t in
     ["dim_data", "dim_torneio", "dim_atleta", "dim_fase", "fato_partida", "fato_atleta_partida"]}

# Chaves primárias únicas
for tabela, pk in [
    ("dim_data", ["id_data"]), ("dim_torneio", ["id_torneio"]),
    ("dim_atleta", ["id_atleta"]), ("dim_fase", ["id_fase"]),
    ("fato_partida", ["id_partida"]), ("fato_atleta_partida", ["id_partida", "id_atleta", "vencedor"]),
]:
    assert g[tabela].select(*pk).distinct().count() == g[tabela].count(), f"{tabela}: PK duplicada"

# Chaves estrangeiras sem órfão: toda FK do fato existe na dimensão
def sem_orfao(fato, fk, dim, pk):
    assert g[fato].join(g[dim], g[fato][fk] == g[dim][pk], "left_anti").count() == 0, f"{fato}.{fk} sem {dim}"

sem_orfao("fato_partida", "id_torneio", "dim_torneio", "id_torneio")
sem_orfao("fato_partida", "id_data", "dim_data", "id_data")
sem_orfao("fato_partida", "id_fase", "dim_fase", "id_fase")
sem_orfao("fato_atleta_partida", "id_partida", "fato_partida", "id_partida")
sem_orfao("fato_atleta_partida", "id_atleta", "dim_atleta", "id_atleta")
sem_orfao("fato_atleta_partida", "id_torneio", "dim_torneio", "id_torneio")
sem_orfao("fato_atleta_partida", "id_data", "dim_data", "id_data")

# Nada foi perdido nem duplicado entre silver e gold
assert g["fato_partida"].count() == p.count(), "fato_partida ≠ silver.partida"
assert g["fato_atleta_partida"].count() == ap.count(), "fato_atleta_partida ≠ silver.atleta_partida"
assert g["dim_atleta"].count() == ap.select("id_atleta").distinct().count(), "dim_atleta ≠ atletas distintos"

# Verificação das flags (contagens medidas no diagnóstico)
fp, fap = g["fato_partida"], g["fato_atleta_partida"]
assert fp.filter("flag_partida_incompleta").count() == 1090, "partidas incompletas ≠ 1.090"
assert fp.filter("flag_duracao_suspeita").count() == 39, "durações suspeitas ≠ 39"
assert fp.filter("flag_placar_inconsistente").count() == 12, "placares inconsistentes ≠ 12"
assert fap.filter("flag_estatistica_invalida").count() == 32, "estatísticas inválidas ≠ 32"
assert fp.filter("sets_vencedor < sets_perdedor and not flag_placar_inconsistente").count() == 0, \
    "placar invertido sem flag"

print(" | ".join(f"{t} {df.count():,}".replace(",", ".") for t, df in g.items()) + " — tudo consistente")